[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/allanspadini/curso-vision-transformers-infnet/blob/main/aula_07_gans_generative_adversarial_networks/aula_07_cyclegan_holo2bright_traducao_dominios.ipynb)

# 🔬 Aula 07 — Caderno Prático 3: Tradução de Domínios Não-Pareada com CycleGAN (Project 9C: Holo2Bright)
### **Faculdade Infnet — Pós-Graduação em Visão Computacional com CNNs e Transformers**
**Professor:** Dr. Allan Spadini  
**Metodologia Pedagógica Central:** Situação-Problema do Mundo Real ➔ Solução de Engenharia ➔ Teoria Rigorosa

---

### 🎯 Objetivos Deste Laboratório:
1. **Compreender o Dilema dos Dados Não-Pareados:** Analisar por que é fisicamente impossível obter pares alinhados pixel a pixel de células biológicas vivas sob microscópios distintos (inviabilizando modelos como Pix2Pix).
2. **Macro-Arquitetura da CycleGAN (Zhu et al., ICCV 2017):** Projetar o sistema quádruplo composto por dois geradores ($G: X \to Y$, $F: Y \to X$) e dois discriminadores PatchGAN ($D_X, D_Y$).
3. **Formalizar a Perda de Consistência de Ciclo (Cycle-Consistency Loss):** Formular e implementar $\mathcal{L}_{cyc}(G, F)$ para garantir bijeção de conteúdo e prevenir colapso de modos sem supervisão pareada.
4. **Engenharia de GPU T4:** Implementar o Replay Buffer de imagens históricas (50 amostras), perda por mínimos quadrados (LSGAN) para gradientes estáveis e geradores ResNet leves de 6 blocos residuais.
5. **Estudo de Caso Prático (Project 9C: Holo2Bright):** Mapear imagens de microscopia holográfica digital (Domínio $X$) para microscopia histológica de campo claro (Domínio $Y$).
6. **Avaliação dos Ciclos Bidirecionais:** Visualizar o ciclo completo de ida e volta ($x \to G(x) \to F(G(x)) \approx x$ e $y \to F(y) \to G(F(y)) \approx y$) e mensurar quantitativamente a preservação morfológica $L_1$.



---
## ⚙️ Parte 1: Configuração do Ambiente, Reprodutibilidade e Acelerador CUDA

Garantimos a fixação de sementes randômicas e inspecionamos os recursos da GPU Nvidia T4 no Google Colab.



In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.utils as vutils

# 1. Reprodutibilidade Científica
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# 2. Detecção Automática de GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Dispositivo Ativo: {device}")
if torch.cuda.is_available():
    print(f"🚀 GPU Detectada: {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"💾 VRAM Total Disponível: {vram_gb:.2f} GB")
else:
    print("⚠️ Atenção: Executando em CPU. Para acelerar o treino, ative a GPU T4 em: Ambiente de Execução > Alterar tipo de ambiente de execução.")



---
## 🔬 Parte 2: O Desafio Biológico dos Dados Não-Pareados & Dataset Holo2Bright

### 1. Situação-Problema do Mundo Real:
A microscopia holográfica digital (Domínio $X$) permite registrar células vivas sem marcadores fluorescentes tóxicos e sem fixação química destrutiva, capturando franjas de difração de fase.
Entretanto, patologistas e algoritmos clínicos de diagnóstico foram calibrados historicamente sobre imagens de microscopia de campo claro (*Bright-Field*, Domínio $Y$).

**Por que o Pix2Pix falha aqui?**
Células vivas movem-se, alteram seu volume citoplasmático e sofrem estresse térmico em segundos. É fisicamente inviável colocar a mesma célula viva exatamente na mesma posição e sob as mesmas condições simultaneamente em dois microscópios físicos distintos. Não possuímos pares $x_i \leftrightarrow y_i$!

### 2. Solução de Engenharia:
Construímos dois conjuntos independentes e não-alinhados:
- **Domínio $X$ (Holografia):** Imagens com franjas de difração de fase e anéis circulares de Airy característicos.
- **Domínio $Y$ (Campo Claro):** Imagens com contraste absorvente de fundo claro e citoplasma/núcleo densos.

Configuramos um pipeline de carregamento programático automático com suporte a dados biomédicos e gerador sintético de morfologias ópticas celulares para garantir execução 100% autônoma.



In [ ]:
# Hiperparâmetros de Dados
IMG_SIZE = 64
BATCH_SIZE = 16

# Transformação: normalização para [-1, 1] compatível com Tanh
img_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

def denormalize(tensor):
    """Converte tensores de [-1, 1] para [0, 1] para visualização."""
    return tensor * 0.5 + 0.5

# Geração Programática Estruturada do Dataset Holo2Bright (Project 9C)
DATASET_DIR = "./holo2bright_data"
os.makedirs(os.path.join(DATASET_DIR, "trainA"), exist_ok=True)  # Domínio X: Holografia
os.makedirs(os.path.join(DATASET_DIR, "trainB"), exist_ok=True)  # Domínio Y: Campo Claro

print("📦 Gerando/Estruturando amostras não-pareadas do Holo2Bright Dataset...")

# Gerar amostras sintéticas de calibração holográfica e campo claro
NUM_SAMPLES = 200

# 1. Domínio X: Padrões de difração e anéis de fase de holografia digital
for i in range(NUM_SAMPLES):
    canvas = np.ones((IMG_SIZE, IMG_SIZE, 3), dtype=np.float32) * 0.2
    yy, xx = np.mgrid[:IMG_SIZE, :IMG_SIZE]
    
    # 2 a 4 células com anéis de difração
    num_cells = np.random.randint(2, 5)
    for _ in range(num_cells):
        cx, cy = np.random.randint(12, IMG_SIZE - 12), np.random.randint(12, IMG_SIZE - 12)
        r = np.sqrt((xx - cx)**2 + (yy - cy)**2)
        # Anéis concêntricos senoidais (difração de holograma)
        fringes = 0.4 * np.sin(r * 0.8) * np.exp(-r / 18.0)
        canvas[:, :, 0] += fringes
        canvas[:, :, 1] += fringes * 0.95
        canvas[:, :, 2] += fringes * 1.1

    canvas = np.clip(canvas, 0.0, 1.0)
    img_pil = Image.fromarray((canvas * 255).astype(np.uint8))
    img_pil.save(os.path.join(DATASET_DIR, "trainA", f"holo_{i:04d}.png"))

# 2. Domínio Y: Imagens histológicas de campo claro (fundo claro, células densas e bordas nítidas)
for i in range(NUM_SAMPLES):
    canvas = np.ones((IMG_SIZE, IMG_SIZE, 3), dtype=np.float32) * 0.88  # Fundo claro característico
    yy, xx = np.mgrid[:IMG_SIZE, :IMG_SIZE]
    
    # Células não pareadas com posições e contagens distintas
    num_cells = np.random.randint(2, 5)
    for _ in range(num_cells):
        cx, cy = np.random.randint(12, IMG_SIZE - 12), np.random.randint(12, IMG_SIZE - 12)
        radius = np.random.randint(6, 11)
        dist = np.sqrt((xx - cx)**2 + (yy - cy)**2)
        # Citoplasma absorvente
        cyto_mask = dist < radius
        canvas[cyto_mask, 0] -= 0.35
        canvas[cyto_mask, 1] -= 0.30
        canvas[cyto_mask, 2] -= 0.15
        # Núcleo denso
        nuc_mask = dist < (radius * 0.45)
        canvas[nuc_mask, 0] -= 0.35
        canvas[nuc_mask, 1] -= 0.40
        canvas[nuc_mask, 2] -= 0.25

    canvas = np.clip(canvas, 0.0, 1.0)
    img_pil = Image.fromarray((canvas * 255).astype(np.uint8))
    img_pil.save(os.path.join(DATASET_DIR, "trainB", f"bright_{i:04d}.png"))

print(f"✅ {NUM_SAMPLES} amostras de Domínio X (Holograma) e {NUM_SAMPLES} de Domínio Y (Campo Claro) criadas com sucesso!")



In [ ]:
class UnpairedDataset(Dataset):
    """Dataset para carregar amostras não-pareadas de dois domínios independentes X e Y."""
    def __init__(self, root_dir, transform=None):
        self.dir_A = os.path.join(root_dir, "trainA")
        self.dir_B = os.path.join(root_dir, "trainB")
        self.files_A = sorted(os.listdir(self.dir_A))
        self.files_B = sorted(os.listdir(self.dir_B))
        self.transform = transform

    def __len__(self):
        return max(len(self.files_A), len(self.files_B))

    def __getitem__(self, idx):
        # Carregar arquivo do Domínio A (Holo)
        file_A = self.files_A[idx % len(self.files_A)]
        img_A = Image.open(os.path.join(self.dir_A, file_A)).convert("RGB")
        
        # Carregar arquivo aleatório do Domínio B (Bright-Field) para garantir desacoplamento
        file_B = self.files_B[random.randint(0, len(self.files_B) - 1)]
        img_B = Image.open(os.path.join(self.dir_B, file_B)).convert("RGB")

        if self.transform:
            img_A = self.transform(img_A)
            img_B = self.transform(img_B)

        return {"A": img_A, "B": img_B}

dataset = UnpairedDataset(DATASET_DIR, transform=img_transforms)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

# Visualizar amostras não pareadas dos dois domínios
sample = next(iter(dataloader))
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
grid_A = vutils.make_grid(denormalize(sample["A"][:8]), nrow=4, padding=2)
grid_B = vutils.make_grid(denormalize(sample["B"][:8]), nrow=4, padding=2)

axes[0].imshow(np.transpose(grid_A.numpy(), (1, 2, 0)))
axes[0].set_title("Domínio X: Microscopia Holográfica (Holo)", fontsize=11, fontweight='bold')
axes[0].axis("off")

axes[1].imshow(np.transpose(grid_B.numpy(), (1, 2, 0)))
axes[1].set_title("Domínio Y: Histologia de Campo Claro (Bright-Field)", fontsize=11, fontweight='bold')
axes[1].axis("off")

plt.suptitle("Amostras Não-Pareadas do Dataset Holo2Bright (Project 9C)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()



---
## 🏗️ Parte 3: Arquitetura da CycleGAN (Geradores ResNet & Discriminadores PatchGAN)

A CycleGAN opera com quatro redes neurais profundas acopladas:
1. **Gerador $G: X \to Y$:** Traduz imagens holográficas em imagens de campo claro.
2. **Gerador $F: Y \to X$:** Traduz imagens de campo claro de volta para holografia.
3. **Discriminador $D_Y$:** Avalia se $G(x)$ parece uma amostra autêntica de campo claro $y$.
4. **Discriminador $D_X$:** Avalia se $F(y)$ parece uma holografia autêntica $x$.

### O Bloco Residual Construtivo:
Utilizamos a arquitetura de **Johnson et al. (2016)** adotada por Zhu et al., composta por convoluções iniciais com stride para downsampling, **6 blocos residuais** intermediários com conexões de atalho $x + \mathcal{F}(x)$, e blocos de convolução transposta para upsampling com ativação `Tanh`.
Em vez de `BatchNorm2d`, utilizamos `InstanceNorm2d`, pois em tradução de estilos de imagens, a normalização independente por canal de cada imagem individual impede artefatos de contraste e vazamento estatístico entre batches.

### O Discriminador PatchGAN $70\times 70$:
Em vez de predizer um único escalar global (o que perde detalhes locais), o PatchGAN classifica patches sobrepostos da imagem, forçando realismo de alta frequência em membranas celulares e texturas finas.



In [ ]:
class ResidualBlock(nn.Module):
    """Bloco Residual com 2 convoluções 3x3, InstanceNorm2d e atalho residual."""
    def __init__(self, channels):
        super(ResidualBlock, self).__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(channels, channels, kernel_size=3, bias=False),
            nn.InstanceNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(channels, channels, kernel_size=3, bias=False),
            nn.InstanceNorm2d(channels)
        )

    def forward(self, x):
        return x + self.block(x)


class ResNetGenerator(nn.Module):
    """Gerador ResNet com 6 blocos residuais para tradução de domínio (Zhu et al., 2017)."""
    def __init__(self, in_channels=3, out_channels=3, num_res_blocks=6):
        super(ResNetGenerator, self).__init__()
        
        # 1. Camada de Entrada (Padding de Reflexão para evitar artefatos de borda)
        model = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(in_channels, 64, kernel_size=7, bias=False),
            nn.InstanceNorm2d(64),
            nn.ReLU(inplace=True)
        ]

        # 2. Downsampling: 64 -> 128 (32x32) -> 256 (16x16)
        in_features = 64
        out_features = in_features * 2
        for _ in range(2):
            model += [
                nn.Conv2d(in_features, out_features, kernel_size=3, stride=2, padding=1, bias=False),
                nn.InstanceNorm2d(out_features),
                nn.ReLU(inplace=True)
            ]
            in_features = out_features
            out_features = in_features * 2

        # 3. Gargalo com 6 Blocos Residuais
        for _ in range(num_res_blocks):
            model += [ResidualBlock(in_features)]

        # 4. Upsampling: 256 -> 128 (32x32) -> 64 (64x64)
        out_features = in_features // 2
        for _ in range(2):
            model += [
                nn.ConvTranspose2d(in_features, out_features, kernel_size=3, stride=2, padding=1, output_padding=1, bias=False),
                nn.InstanceNorm2d(out_features),
                nn.ReLU(inplace=True)
            ]
            in_features = out_features
            out_features = in_features // 2

        # 5. Camada de Saída com Tanh
        model += [
            nn.ReflectionPad2d(3),
            nn.Conv2d(64, out_channels, kernel_size=7),
            nn.Tanh()
        ]

        self.model = nn.Sequential(*model)

    def forward(self, x):
        return self.model(x)


class PatchGANDiscriminator(nn.Module):
    """Discriminador PatchGAN que classifica o realismo em patches locais da imagem."""
    def __init__(self, in_channels=3):
        super(PatchGANDiscriminator, self).__init__()

        def discriminator_block(in_f, out_f, normalize=True):
            layers = [nn.Conv2d(in_f, out_f, kernel_size=4, stride=2, padding=1)]
            if normalize:
                layers.append(nn.InstanceNorm2d(out_f))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers

        self.model = nn.Sequential(
            *discriminator_block(in_channels, 64, normalize=False),
            *discriminator_block(64, 128),
            *discriminator_block(128, 256),
            nn.Conv2d(256, 1, kernel_size=4, stride=1, padding=1)  # Mapa de predição de patches (LSGAN)
        )

    def forward(self, x):
        return self.model(x)



In [ ]:
# Instanciar os 2 Geradores e os 2 Discriminadores
G_AB = ResNetGenerator(in_channels=3, out_channels=3, num_res_blocks=6).to(device)  # Holo -> Bright
G_BA = ResNetGenerator(in_channels=3, out_channels=3, num_res_blocks=6).to(device)  # Bright -> Holo
D_A = PatchGANDiscriminator(in_channels=3).to(device)                               # Valida Holo (Domínio X)
D_B = PatchGANDiscriminator(in_channels=3).to(device)                               # Valida Bright (Domínio Y)

def weights_init_normal(m):
    classname = m.__class__.__name__
    if classname.find("Conv") != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find("InstanceNorm2d") != -1:
        if m.weight is not None:
            nn.init.normal_(m.weight.data, 1.0, 0.02)
        if m.bias is not None:
            nn.init.constant_(m.bias.data, 0.0)

G_AB.apply(weights_init_normal)
G_BA.apply(weights_init_normal)
D_A.apply(weights_init_normal)
D_B.apply(weights_init_normal)

# Contagem de parâmetros treináveis
total_G = sum(p.numel() for p in G_AB.parameters()) * 2
total_D = sum(p.numel() for p in D_A.parameters()) * 2

print(f"🧠 Total nos 2 Geradores (G_AB + G_BA):     {total_G:,} parâmetros")
print(f"🛡️ Total nos 2 Discriminadores (D_A + D_B): {total_D:,} parâmetros")

# Teste de Sanidade de Shapes Tensoriais
dummy_A = torch.randn(2, 3, IMG_SIZE, IMG_SIZE, device=device)
fake_B = G_AB(dummy_A)
rec_A = G_BA(fake_B)
pred_D = D_B(fake_B)

print(f"\n🧪 Sanity Check:")
print(f"  Entrada x (Holo):          {list(dummy_A.shape)}")
print(f"  Traduzida G(x) (Bright):   {list(fake_B.shape)}")
print(f"  Reconstruída F(G(x)):      {list(rec_A.shape)}")
print(f"  Mapa PatchGAN de D_B:      {list(pred_D.shape)}")
assert fake_B.shape == (2, 3, IMG_SIZE, IMG_SIZE), "Erro no shape do Gerador!"
print("  ✅ Rastreamento Tensorial da CycleGAN Validado com Sucesso!")



---
## ⚖️ Parte 4: Engenharia de Perdas da CycleGAN & Replay Buffer

O objetivo de otimização total da CycleGAN é composto por três componentes fundamentais:

$$\mathcal{L}(G, F, D_X, D_Y) = \mathcal{L}_{\text{adv}}(G, D_Y) + \mathcal{L}_{\text{adv}}(F, D_X) + \lambda_{\text{cyc}} \mathcal{L}_{\text{cyc}}(G, F) + \lambda_{\text{idt}} \mathcal{L}_{\text{idt}}(G, F)$$

### 1. Perdas Adversariais (LSGAN - Mean Squared Error):
Em vez de Binary Cross-Entropy (que pode saturar quando o discriminador vence), Zhu et al. utilizam a formulação de **Mao et al. (LSGAN)** com erro médio quadrático:
$$\mathcal{L}_{\text{adv}}(G, D_Y) = \mathbb{E}_{x} [(D_Y(G(x)) - 1)^2]$$

### 2. Perda de Consistência de Ciclo (Cycle-Consistency Loss):
Impede que o gerador sofra colapso de modos ou alucine células aleatórias, penalizando desvios na norma $L_1$:
$$\mathcal{L}_{\text{cyc}}(G, F) = \mathbb{E}_{x} [\|F(G(x)) - x\|_1] + \mathbb{E}_{y} [\|G(F(y)) - y\|_1]$$
Ponderada com $\lambda_{\text{cyc}} = 10.0$.

### 3. Perda de Identidade (Identity Loss):
Garante que se uma imagem que já pertence ao domínio alvo for fornecida ao gerador, ela seja preservada sem alterações cromáticas espúrias:
$$\mathcal{L}_{\text{idt}}(G, F) = \mathbb{E}_{y} [\|G(y) - y\|_1] + \mathbb{E}_{x} [\|F(x) - x\|_1]$$
Ponderada com $\lambda_{\text{idt}} = 5.0$ ($0.5 \times \lambda_{\text{cyc}}$).

### 4. Replay Buffer de Imagens Históricas:
Armazenamos as últimas 50 imagens sintéticas geradas. Ao atualizar $D_A$ e $D_B$, sorteamos 50% das amostras falsas desse histórico para mitigar oscilações extremas e esquecimento catastrófico no equilíbrio de Nash.



In [ ]:
class ReplayBuffer:
    """Buffer que armazena imagens geradas anteriormente para estabilizar os discriminadores."""
    def __init__(self, max_size=50):
        self.max_size = max_size
        self.data = []

    def push_and_pop(self, data):
        to_return = []
        for element in data.data:
            element = torch.unsqueeze(element, 0)
            if len(self.data) < self.max_size:
                self.data.append(element)
                to_return.append(element)
            else:
                if random.uniform(0, 1) > 0.5:
                    i = random.randint(0, self.max_size - 1)
                    to_return.append(self.data[i].clone())
                    self.data[i] = element
                else:
                    to_return.append(element)
        return torch.cat(to_return, 0)

# Instanciar buffers para os dois geradores
fake_A_buffer = ReplayBuffer(max_size=50)
fake_B_buffer = ReplayBuffer(max_size=50)

# Funções de perda
criterion_GAN = nn.MSELoss()   # Perda LSGAN para estabilidade
criterion_cycle = nn.L1Loss()  # Perda de ciclo L1
criterion_identity = nn.L1Loss() # Perda de identidade L1

# Hiperparâmetros de Pesos das Perdas (Zhu et al.)
LAMBDA_CYC = 10.0
LAMBDA_IDT = 5.0

# Otimizadores Adam (LR=0.0002, Betas=(0.5, 0.999))
import itertools
optimizer_G = optim.Adam(
    itertools.chain(G_AB.parameters(), G_BA.parameters()),
    lr=0.0002, betas=(0.5, 0.999)
)
optimizer_D_A = optim.Adam(D_A.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizer_D_B = optim.Adam(D_B.parameters(), lr=0.0002, betas=(0.5, 0.999))

print("🎯 Configuração de Otimização e Perdas Concluída!")



In [ ]:
NUM_EPOCHS = 15

history = {
    'loss_G': [],
    'loss_G_adv': [],
    'loss_cycle': [],
    'loss_identity': [],
    'loss_D': []
}

# Amostras fixas para acompanhar a evolução visual durante as épocas
fixed_sample = next(iter(dataloader))
fixed_real_A = fixed_sample["A"][:4].to(device)
fixed_real_B = fixed_sample["B"][:4].to(device)

print(f"🚀 Iniciando Treinamento da CycleGAN ({NUM_EPOCHS} Épocas)...")
print("=" * 70)

for epoch in range(1, NUM_EPOCHS + 1):
    loop = tqdm(dataloader, desc=f"Época [{epoch:02d}/{NUM_EPOCHS:02d}]", leave=True)
    
    epoch_loss_G = 0.0
    epoch_loss_G_adv = 0.0
    epoch_loss_cycle = 0.0
    epoch_loss_identity = 0.0
    epoch_loss_D = 0.0

    for batch in loop:
        real_A = batch["A"].to(device)
        real_B = batch["B"].to(device)

        # =====================================================================
        # PASSO 1: Atualização dos Geradores G_AB e G_BA
        # =====================================================================
        optimizer_G.zero_grad()

        # 1.1 Perda de Identidade: G_AB(B) ≈ B e G_BA(A) ≈ A
        loss_id_A = criterion_identity(G_BA(real_A), real_A)
        loss_id_B = criterion_identity(G_AB(real_B), real_B)
        loss_identity = (loss_id_A + loss_id_B) / 2.0

        # 1.2 Perda Adversarial: G_AB deve enganar D_B e G_BA deve enganar D_A
        fake_B = G_AB(real_A)
        pred_fake_B = D_B(fake_B)
        target_real_B = torch.ones_like(pred_fake_B)
        loss_GAN_AB = criterion_GAN(pred_fake_B, target_real_B)

        fake_A = G_BA(real_B)
        pred_fake_A = D_A(fake_A)
        target_real_A = torch.ones_like(pred_fake_A)
        loss_GAN_BA = criterion_GAN(pred_fake_A, target_real_A)

        loss_GAN = (loss_GAN_AB + loss_GAN_BA) / 2.0

        # 1.3 Perda de Consistência de Ciclo: G_BA(G_AB(A)) ≈ A e G_AB(G_BA(B)) ≈ B
        recov_A = G_BA(fake_B)
        loss_cycle_A = criterion_cycle(recov_A, real_A)

        recov_B = G_AB(fake_A)
        loss_cycle_B = criterion_cycle(recov_B, real_B)

        loss_cycle = (loss_cycle_A + loss_cycle_B) / 2.0

        # Perda Total do Gerador
        loss_G = loss_GAN + LAMBDA_CYC * loss_cycle + LAMBDA_IDT * loss_identity
        loss_G.backward()
        optimizer_G.step()

        # =====================================================================
        # PASSO 2: Atualização do Discriminador D_A (Avalia Holo x vs fake_A)
        # =====================================================================
        optimizer_D_A.zero_grad()

        pred_real_A = D_A(real_A)
        loss_D_real_A = criterion_GAN(pred_real_A, torch.ones_like(pred_real_A))

        fake_A_buffered = fake_A_buffer.push_and_pop(fake_A)
        pred_fake_A_buf = D_A(fake_A_buffered.detach())
        loss_D_fake_A = criterion_GAN(pred_fake_A_buf, torch.zeros_like(pred_fake_A_buf))

        loss_D_A = (loss_D_real_A + loss_D_fake_A) / 2.0
        loss_D_A.backward()
        optimizer_D_A.step()

        # =====================================================================
        # PASSO 3: Atualização do Discriminador D_B (Avalia Bright y vs fake_B)
        # =====================================================================
        optimizer_D_B.zero_grad()

        pred_real_B = D_B(real_B)
        loss_D_real_B = criterion_GAN(pred_real_B, torch.ones_like(pred_real_B))

        fake_B_buffered = fake_B_buffer.push_and_pop(fake_B)
        pred_fake_B_buf = D_B(fake_B_buffered.detach())
        loss_D_fake_B = criterion_GAN(pred_fake_B_buf, torch.zeros_like(pred_fake_B_buf))

        loss_D_B = (loss_D_real_B + loss_D_fake_B) / 2.0
        loss_D_B.backward()
        optimizer_D_B.step()

        loss_D = (loss_D_A + loss_D_B) / 2.0

        # Acúmulo de métricas
        epoch_loss_G += loss_G.item()
        epoch_loss_G_adv += loss_GAN.item()
        epoch_loss_cycle += loss_cycle.item()
        epoch_loss_identity += loss_identity.item()
        epoch_loss_D += loss_D.item()

        loop.set_postfix({
            'Loss_G': f"{loss_G.item():.2f}",
            'Cyc': f"{loss_cycle.item():.3f}",
            'Idt': f"{loss_identity.item():.3f}",
            'Loss_D': f"{loss_D.item():.3f}"
        })

    n_b = len(dataloader)
    history['loss_G'].append(epoch_loss_G / n_b)
    history['loss_G_adv'].append(epoch_loss_G_adv / n_b)
    history['loss_cycle'].append(epoch_loss_cycle / n_b)
    history['loss_identity'].append(epoch_loss_identity / n_b)
    history['loss_D'].append(epoch_loss_D / n_b)

print("=" * 70)
print("🎉 Treinamento da CycleGAN Finalizado com Sucesso!")



---
## 📊 Parte 5: Diagnóstico e Curvas de Convergência da CycleGAN

Inspecionamos a evolução das perdas durante o treinamento:
1. **Perda de Consistência de Ciclo (Cycle Loss):** Deve apresentar queda contínua, indicando que o modelo está aprendendo a preservar a morfologia e localização celular exatas no ciclo de ida e volta.
2. **Perda de Identidade (Identity Loss):** Deve convergir para valores baixos, confirmando a estabilização das cores e contraste óptico.
3. **Perdas Adversariais ($G$ e $D$):** Devem manter dinâmica de oscilação estável sem colapso de nenhum dos lados.



In [ ]:
plt.figure(figsize=(15, 4.5))

plt.subplot(1, 3, 1)
plt.plot(history['loss_G'], label='Loss Total G', color='#0284C7', lw=2)
plt.plot(history['loss_D'], label='Loss Total D', color='#E65100', lw=2)
plt.title('Dinâmica Global G vs D', fontsize=11, fontweight='bold')
plt.xlabel('Época')
plt.ylabel('Loss')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(history['loss_cycle'], label='Loss Ciclo (L1)', color='#16A34A', lw=2)
plt.plot(history['loss_identity'], label='Loss Identidade (L1)', color='#9333EA', lw=2)
plt.title('Preservação de Conteúdo (Ciclo & Identidade)', fontsize=11, fontweight='bold')
plt.xlabel('Época')
plt.ylabel('Loss L1')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(history['loss_G_adv'], label='Perda Adversarial G (LSGAN)', color='#DC2626', lw=2)
plt.title('Realismo Adversarial', fontsize=11, fontweight='bold')
plt.xlabel('Época')
plt.ylabel('Loss MSE')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

plt.tight_layout()
plt.show()



---
## 🔬 Parte 6: Avaliação Bidirecional Completa dos Ciclos (Holo2Bright)

Visualizamos na prática os dois ciclos da CycleGAN:
1. **Ciclo Forward ($X \to Y \to X$):**
   $$\text{Holograma Real } x \xrightarrow{G_{AB}} \text{Campo Claro Traduzido } G(x) \xrightarrow{G_{BA}} \text{Holograma Reconstruído } F(G(x)) \approx x$$
2. **Ciclo Backward ($Y \to X \to Y$):**
   $$\text{Campo Claro Real } y \xrightarrow{G_{BA}} \text{Holograma Traduzido } F(y) \xrightarrow{G_{AB}} \text{Campo Claro Reconstruído } G(F(y)) \approx y$$



In [ ]:
G_AB.eval()
G_BA.eval()

with torch.no_grad():
    # 1. Ciclo Forward: x -> G_AB(x) -> G_BA(G_AB(x))
    fake_B_eval = G_AB(fixed_real_A)
    recov_A_eval = G_BA(fake_B_eval)

    # 2. Ciclo Backward: y -> G_BA(y) -> G_AB(G_BA(y))
    fake_A_eval = G_BA(fixed_real_B)
    recov_B_eval = G_AB(fake_A_eval)

# Calcular métricas de erro L1 de reconstrução de ciclo
l1_forward = nn.L1Loss()(recov_A_eval, fixed_real_A).item()
l1_backward = nn.L1Loss()(recov_B_eval, fixed_real_B).item()

print(f"📏 Erro Médio de Ciclo Forward  ||F(G(x)) - x||_1: {l1_forward:.4f}")
print(f"📏 Erro Médio de Ciclo Backward ||G(F(y)) - y||_1: {l1_backward:.4f}")

# Plotagem do Tríptico do Ciclo Forward (Holo -> Bright -> Holo)
fig, axes = plt.subplots(3, 4, figsize=(14, 10))

for i in range(4):
    # Coluna i: Imagem Real x
    axes[0, i].imshow(np.transpose(denormalize(fixed_real_A[i]).cpu().numpy(), (1, 2, 0)))
    axes[0, i].set_title(f"1. Real x (Holo #{i+1})", fontsize=10, fontweight='bold', color='#0369A1')
    axes[0, i].axis("off")

    # Coluna i: Imagem Traduzida G(x)
    axes[1, i].imshow(np.transpose(denormalize(fake_B_eval[i]).cpu().numpy(), (1, 2, 0)))
    axes[1, i].set_title(f"2. G(x) (Campo Claro)", fontsize=10, fontweight='bold', color='#15803D')
    axes[1, i].axis("off")

    # Coluna i: Imagem Reconstruída F(G(x))
    axes[2, i].imshow(np.transpose(denormalize(recov_A_eval[i]).cpu().numpy(), (1, 2, 0)))
    axes[2, i].set_title(f"3. F(G(x)) (Reconstruída)", fontsize=10, fontweight='bold', color='#C2410C')
    axes[2, i].axis("off")

plt.suptitle("Ciclo Forward da CycleGAN: Preservação Morfológica Holo ➔ Bright ➔ Holo", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()



In [ ]:
# Plotagem do Tríptico do Ciclo Backward (Bright -> Holo -> Bright)
fig, axes = plt.subplots(3, 4, figsize=(14, 10))

for i in range(4):
    # Coluna i: Imagem Real y
    axes[0, i].imshow(np.transpose(denormalize(fixed_real_B[i]).cpu().numpy(), (1, 2, 0)))
    axes[0, i].set_title(f"1. Real y (Bright #{i+1})", fontsize=10, fontweight='bold', color='#15803D')
    axes[0, i].axis("off")

    # Coluna i: Imagem Traduzida F(y)
    axes[1, i].imshow(np.transpose(denormalize(fake_A_eval[i]).cpu().numpy(), (1, 2, 0)))
    axes[1, i].set_title(f"2. F(y) (Holograma)", fontsize=10, fontweight='bold', color='#0369A1')
    axes[1, i].axis("off")

    # Coluna i: Imagem Reconstruída G(F(y))
    axes[2, i].imshow(np.transpose(denormalize(recov_B_eval[i]).cpu().numpy(), (1, 2, 0)))
    axes[2, i].set_title(f"3. G(F(y)) (Reconstruída)", fontsize=10, fontweight='bold', color='#C2410C')
    axes[2, i].axis("off")

plt.suptitle("Ciclo Backward da CycleGAN: Preservação Morfológica Bright ➔ Holo ➔ Bright", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()



---
## 💡 Parte 7: Conclusões, Boas Práticas e Exercícios Propostos

### 🎓 O Que Aprendemos Neste Laboratório:
1. **Tradução Não-Supervisionada:** Conseguimos mapear entre modalidades ópticas radicalmente distintas (franjas de fase holográficas vs contraste de campo claro) **sem nenhum par alinhado** da mesma célula.
2. **O Papel Crítico de $\mathcal{L}_{cyc}$:** A restrição biunívoca de ida e volta impede que o gerador colapse em uma única imagem estática, forçando a preservação estrita da geometria celular original.
3. **Engenharia de Estabilidade:** A combinação de **PatchGAN**, **LSGAN** (erro quadrático) e **Replay Buffer** garantiu convergência estável em GPU T4 sem divergência no equilíbrio de Nash.

---

### 🧠 Desafios e Exercícios para Pós-Graduação:
1. **Impacto do Hiperparâmetro $\lambda_{\text{cyc}}$:**
   - Experimente alterar `LAMBDA_CYC` para $0.0$ (desativando a consistência de ciclo). Observe como o gerador perde o compromisso de manter o arranjo celular de entrada, gerando distorções morfológicas severas.
2. **Impacto da Perda de Identidade ($\lambda_{\text{idt}}$):**
   - Treine a rede com `LAMBDA_IDT = 0.0` e analise se o gerador $G_{AB}$ altera indevidamente o fundo ou o tom de imagens que já são de campo claro.
3. **Integração Downstream com o Projeto da Disciplina:**
   - Utilize as imagens de campo claro traduzidas $G(x)$ como dados de entrada para um classificador ResNet pré-treinado no Domínio $Y$ e avalie o ganho na sensibilidade de detecção em relação a classificar diretamente sobre o holograma bruto sem alinhamento de domínio.

